<a href="https://colab.research.google.com/github/ravneetkour/python/blob/main/Video_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **YOLO-Kitti-Vehicle-Detection**

**Important Note:**

> Toggle the GPU runtime on





---



# 1. Install ultralytics

In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

from ultralytics import YOLO

Ultralytics 8.4.59 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.2/112.6 GB disk)


# 2. Initialize the YOLO model

In [ ]:
model = YOLO("yolo26n.pt")

# 3. Train the model (Automatically uses GPU if available!)
### Sit back for about 15 minutes, let it finish training, and enjoy your fully annotated KITTI video!

In [ ]:
# Setting workers=4 to prevent CPU thread choking
results = model.train(data="kitti.yaml", epochs=10, imgsz=640, workers=4)

Ultralytics 8.4.59 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=kitti.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

# 4. Load your newly fine-tuned model for inference

In [ ]:
# model.trainer.save_dir points to where the weights were just saved
modelp = YOLO(f"{model.trainer.save_dir}/weights/best.pt")

# Upload model to HuggingFace Directly from Google Colab

**Note:** HF TOKEN is Required

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import HfApi, login
from google.colab import userdata

# 1. Get your token and authenticate
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab secrets!")
login(token=HF_TOKEN)

# 2. Define your repository details
# Change 'your-username' to your actual Hugging Face username
repo_id = "missravneetkaur/KITTI-Vision-YOLO"

# 3. Initialize the Hugging Face API
api = HfApi()

# 4. Create the repository on the Hub (if it doesn't exist yet)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 5. Define the path to your best weights
# This dynamically grabs the path from your training run
weights_path = f"{model.trainer.save_dir}/weights/best.pt"

# 6. Upload the file
print(f"Uploading {weights_path} to Hugging Face Hub...")
api.upload_file(
    path_or_fileobj=weights_path,
    path_in_repo="best.pt", # How it will be named inside your HF repo
    repo_id=repo_id,
    repo_type="model"
)
print(f"Successfully uploaded! Check it out here: https://huggingface.co/{repo_id}")

Uploading /content/runs/detect/train/weights/best.pt to Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ect/train/weights/best.pt:  10%|#         |  540kB / 5.35MB            

Successfully uploaded! Check it out here: https://huggingface.co/missravneetkaur/KITTI-Vision-YOLO


**Model Link on HF:** https://huggingface.co/missravneetkaur/KITTI-Vision-YOLO



---





---





---



# **1. Download and Load the Model from HuggingFace**

In [ ]:
!uv pip install ultralytics -q
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# 1. Download the weights file from your Hugging Face repo
# Replace 'your-username' with your actual HF username
local_weights_path = hf_hub_download(
    repo_id="missravneetkaur/KITTI-Vision-YOLO",
    filename="best.pt"
)

# 2. Load the downloaded file into YOLO
modelp = YOLO(local_weights_path)
print("Model loaded successfully from Hugging Face Hub!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


best.pt:   0%|          | 0.00/5.35M [00:00<?, ?B/s]

Model loaded successfully from Hugging Face Hub!


# 5. Run inference on a sample video and SAVE the results

In [ ]:
# This will save an annotated MP4 video to 'runs/detect/predict/'
prediction_results = modelp.predict(
    source="https://ultralytics.com/assets/kitti-inference-vid.mp4",
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/498) /content/kitti-inference-vid.mp4: 224x640 6 cars, 71.5ms
video 1/1 (frame 2/498) /content/kitti-inference-vid.mp4: 224x640 6 cars, 11.3ms
video 1/1 (frame 3/498) /content/kitti-inference-vid.mp4: 224x640 6 cars, 10.6ms
video 1/1 (frame 4/498) /content/kitti-inference-vid.mp4: 224x640 6 cars, 11.1ms
video 1/1 (frame 5/498) /content/kitti-inference-vid.mp4: 224x640 6 cars, 10.6ms
video 1/1 (frame 6/498) /content/kitti-inference-vid.

**or**

In [ ]:
# Inference using the model on a single image
prediction_results = modelp.predict(
    source="https://ultralytics.com/assets/kitti-inference-im0.png",  # URL or local path to image
    save=True                                                         # Saves the annotated image
)


image 1/1 /content/kitti-inference-im0.png: 224x640 12 cars, 1 truck, 10.7ms
Speed: 1.3ms preprocess, 10.7ms inference, 0.4ms postprocess per image at shape (1, 3, 224, 640)
Results saved to /content/runs/detect/predict


*Made with 💗*



---

